# DriveSense-VLM — 02: Model Optimization

Runs the full 3-stage optimization pipeline:
1. **LoRA merge** — fold adapter weights into the base model (bfloat16)
2. **AWQ 4-bit quantization** — LLM decoder only; ViT stays fp16
3. **TensorRT ViT compilation** — ONNX export + FP16 TRT engine

**GPU required**: A100 80GB  
**Estimated time**: ~1.5 hours  
**Estimated cost**: ~18 CU  

**Prerequisites**: `01_training.ipynb` must have produced a checkpoint in
`outputs/training/checkpoint-best` (or latest `checkpoint-*`).

## ⚠️ Before Running

Set **Runtime → Change runtime type → A100 GPU** before starting.
TensorRT compilation requires CUDA; AWQ quantization needs ≥ 40 GB VRAM.

In [ ]:
# Cell 3: Mount Drive + clone repo + symlinks + install deps
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/DriveSense-VLM'
os.makedirs(PROJECT_ROOT, exist_ok=True)

!git clone https://github.com/jayanth922/DriveSense-VLM.git /content/DriveSense-VLM 2>/dev/null || (cd /content/DriveSense-VLM && git pull --quiet)
%cd /content/DriveSense-VLM

!mkdir -p {PROJECT_ROOT}/data {PROJECT_ROOT}/outputs
!ln -sf {PROJECT_ROOT}/data /content/DriveSense-VLM/data 2>/dev/null || true
!ln -sf {PROJECT_ROOT}/outputs /content/DriveSense-VLM/outputs 2>/dev/null || true

# Upgrade setuptools first (Colab ships an outdated version that breaks PEP 660 editable installs)
!pip install --upgrade setuptools wheel build -q

# Install inference deps — editable preferred, non-editable fallback
!pip install -e '.[inference]' -q 2>/dev/null || pip install '.[inference]' -q
!pip install autoawq -q
# TensorRT — try install, graceful fallback to torch.compile if unavailable
!pip install tensorrt -q 2>/dev/null && echo 'TensorRT installed.' || echo 'TensorRT unavailable — will use torch.compile fallback.'

print('Setup complete.')

In [ ]:
# Cell 4: Verify GPU + check training output exists
import torch, glob
from pathlib import Path

assert torch.cuda.is_available(), (
    'No GPU! Set Runtime → Change runtime type → A100 GPU'
)
print(f'GPU  : {torch.cuda.get_device_name(0)}')
print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Locate adapter checkpoint
training_dir = Path(PROJECT_ROOT) / 'outputs' / 'training'
best = training_dir / 'checkpoint-best'
if not best.exists():
    ckpts = sorted(glob.glob(str(training_dir / 'checkpoint-*')))
    if not ckpts:
        raise FileNotFoundError(
            f'No checkpoints in {training_dir}.\n'
            'Run 01_training.ipynb first.'
        )
    best = Path(ckpts[-1])

ADAPTER_PATH = str(best)
print(f'\nAdapter path : {ADAPTER_PATH} ✓')

In [ ]:
# Cell 5: Run full optimization pipeline (merge → quantize → TensorRT)
# Each stage is idempotent — skips if sentinel file already exists.
# RECOVERY: If session disconnects, rerun Cells 3–4, then rerun this cell.
#           Completed stages will be skipped automatically.
!python scripts/run_optimize_model.py --all \
    --adapter-path {ADAPTER_PATH} \
    --override inference.merge.output_dir={PROJECT_ROOT}/outputs/merged_model \
    --override inference.quantization.output_dir={PROJECT_ROOT}/outputs/quantized_model \
    --override inference.tensorrt.output_dir={PROJECT_ROOT}/outputs/tensorrt

In [ ]:
# Cell 6: Display optimization results
import json
from pathlib import Path

print('=' * 60)
print('  Optimization Results')
print('=' * 60)

# Merged model
merged = Path(PROJECT_ROOT) / 'outputs' / 'merged_model'
if merged.exists():
    size = sum(f.stat().st_size for f in merged.rglob('*') if f.is_file()) / 1e9
    print(f'  Merged model   : {merged}  ({size:.2f} GB)')

# Quantized model
quant = Path(PROJECT_ROOT) / 'outputs' / 'quantized_model'
if quant.exists():
    size = sum(f.stat().st_size for f in quant.rglob('*') if f.is_file()) / 1e9
    cfg = quant / 'quant_config.json'
    bits = json.loads(cfg.read_text()).get('bits', '?') if cfg.exists() else '?'
    print(f'  Quantized model: {quant}  ({size:.2f} GB, AWQ {bits}-bit)')

# TensorRT engine
trt = Path(PROJECT_ROOT) / 'outputs' / 'tensorrt'
if trt.exists():
    engine = trt / 'vit.engine'
    fallback = trt / 'fallback_info.json'
    if engine.exists():
        print(f'  TensorRT engine: {engine}  ({engine.stat().st_size / 1e6:.1f} MB)')
    elif fallback.exists():
        fb = json.loads(fallback.read_text())
        print(f'  TRT fallback   : {fb.get("method", "unknown")} (see {fallback})')

print()

In [ ]:
# Cell 7: Verify all outputs exist on Drive
from pathlib import Path

checks = {
    'Merged model config': Path(PROJECT_ROOT) / 'outputs' / 'merged_model' / 'config.json',
    'AWQ quant config'   : Path(PROJECT_ROOT) / 'outputs' / 'quantized_model' / 'quant_config.json',
    'TRT report'         : Path(PROJECT_ROOT) / 'outputs' / 'tensorrt' / 'optimization_report.txt',
}

all_ok = True
for label, path in checks.items():
    status = '✓' if path.exists() else '✗ MISSING'
    print(f'  {label:<30}: {status}')
    if not path.exists():
        all_ok = False

if all_ok:
    print('\nAll optimization outputs present. Proceed to 03_benchmark.ipynb')
else:
    print('\nSome outputs missing — check the optimization log above.')